# Training a neural network model

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.exceptions import UndefinedMetricWarning
from datatypes import CLEAN_SQF
import warnings

# Silence warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Main analysis
### Split dataset into training and testing sets

In [27]:
clean_df = pd.read_csv(CLEAN_SQF)

features = clean_df.columns.difference(["USE_OF_FORCE_FLAG"]).values
X = clean_df[features]
y = clean_df["USE_OF_FORCE_FLAG"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123
)

# Convert train and test subsets into PyTorch tensors
X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test = torch.tensor(X_test.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

In [28]:
num_features = X_train.shape[1]

In [ ]:
### WIP -- currently referencing https://machinelearningmastery.com/develop-your-first-neural-network-with-pytorch-step-by-step/

model = nn.Sequential(
    nn.Linear(num_features, 16), ### CHECK HOW TO DETERMINE THE BEST SECOND VALUE - if it is a hyperparameter
    nn.ReLU(),
    nn.Linear(16, num_features),
    nn.ReLU(),
    nn.Linear(num_features, 1),
    nn.Sigmoid()
)

In [36]:
bce_loss = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [37]:
n_epochs = 100
batch_size = 10

for epoch in range(n_epochs):
    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]
        y_pred = model(X_batch)
        y_batch = y_train[i:i+batch_size]
        loss = bce_loss(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Finished epoch {epoch}, latest loss {loss}")

Finished epoch 0, latest loss 0.18488264083862305
Finished epoch 1, latest loss 0.18947473168373108
Finished epoch 2, latest loss 0.1934969127178192
Finished epoch 3, latest loss 0.19784723222255707
Finished epoch 4, latest loss 0.2049863338470459
Finished epoch 5, latest loss 0.20304611325263977
Finished epoch 6, latest loss 0.20411467552185059
Finished epoch 7, latest loss 0.2025517225265503
Finished epoch 8, latest loss 0.20651774108409882
Finished epoch 9, latest loss 0.20695412158966064
Finished epoch 10, latest loss 0.20690052211284637
Finished epoch 11, latest loss 0.20539391040802002
Finished epoch 12, latest loss 0.20778529345989227
Finished epoch 13, latest loss 0.20788896083831787
Finished epoch 14, latest loss 0.21562883257865906
Finished epoch 15, latest loss 0.2211398482322693
Finished epoch 16, latest loss 0.21840783953666687
Finished epoch 17, latest loss 0.22360925376415253
Finished epoch 18, latest loss 0.22822125256061554
Finished epoch 19, latest loss 0.223677873611

In [42]:
# Compute training accuracy
with torch.no_grad():
    y_pred = model(X_train)
accuracy = (y_pred.round() == y_train).float().mean()
print(f"Accuracy {accuracy}")

Accuracy 0.7949978709220886


In [43]:
# Compute testing accuracy
with torch.no_grad():
    y_pred = model(X_test)
accuracy = (y_pred.round() == y_test).float().mean()
print(f"Accuracy {accuracy}")

Accuracy 0.769759476184845
